In [ ]:
# GEO query and DEG analysis script

# Install dependencies if not already installed:
!pip install GEOparse pandas matplotlib seaborn scipy numpy

import GEOparse
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt
import seaborn as sns
import GEOparse
import pandas as pd
gse_id = "GSE212865"  # <-- Change GEO Series
gse = GEOparse.get_GEO(geo=gse_id, destdir=".")

print(f"Dataset: {gse_id}")
print(f"Title: {gse.metadata['title'][0]}")
print(f"Samples: {len(gse.gsms)}")
dfs = []
for gsm_name, gsm in gse.gsms.items():
    if gsm.table is not None and "VALUE" in gsm.table.columns:
        df = gsm.table[["ID_REF", "VALUE"]].copy()
        df.rename(columns={"VALUE": gsm_name}, inplace=True)
        dfs.append(df)

expr_df = dfs[0]
for df in dfs[1:]:
    expr_df = expr_df.merge(df, on="ID_REF")

expr_df = expr_df.drop_duplicates(subset="ID_REF")
expr_df.set_index("ID_REF", inplace=True)

print("Expression Matrix Shape:", expr_df.shape)
expr_df.to_csv("expression.csv")
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

n = len(expr_df.columns)
control = expr_df.iloc[:, :n//2]
treated = expr_df.iloc[:, n//2:]

pvals, logFCs, t_stats, avg_exp_values = [], [], [], []

for gene in expr_df.index:
    control_values = control.loc[gene]
    treated_values = treated.loc[gene]

    if control_values.dropna().empty or treated_values.dropna().empty:
        stat, p, logFC, avg_exp = np.nan, np.nan, np.nan, np.nan
    else:
        stat, p = ttest_ind(control_values, treated_values, nan_policy="omit")
        logFC = treated_values.mean() - control_values.mean()
        avg_exp = expr_df.loc[gene].mean()

    t_stats.append(stat)
    pvals.append(p)
    logFCs.append(logFC)
    avg_exp_values.append(avg_exp)

deg_df = pd.DataFrame({
    "Gene": expr_df.index,
    "logFC": logFCs,
    "t": t_stats,
    "Pvalue": pvals,
    "AvgExp": avg_exp_values
})

# Safe log10
deg_df["-log10p"] = -np.log10(deg_df["Pvalue"].replace(0, np.nan))

# Adjust p-values
valid = deg_df["Pvalue"].notna()
if valid.sum() > 1:
    deg_df.loc[valid, "adjPvalue"] = multipletests(deg_df.loc[valid, "Pvalue"], method='fdr_bh')[1]
else:
    deg_df["adjPvalue"] = np.nan

deg_df.to_csv("DEG.csv", index=False)
print("✅ Saved: DEG.csv")

10-Apr-2026 13:32:25 DEBUG utils - Directory . already exists. Skipping.
DEBUG:GEOparse:Directory . already exists. Skipping.
10-Apr-2026 13:32:25 INFO GEOparse - File already exist: using local version.
INFO:GEOparse:File already exist: using local version.
10-Apr-2026 13:32:25 INFO GEOparse - Parsing ./GSE212865_family.soft.gz: 
INFO:GEOparse:Parsing ./GSE212865_family.soft.gz: 
10-Apr-2026 13:32:25 DEBUG GEOparse - DATABASE: GeoMiame
DEBUG:GEOparse:DATABASE: GeoMiame
10-Apr-2026 13:32:25 DEBUG GEOparse - SERIES: GSE212865
DEBUG:GEOparse:SERIES: GSE212865
10-Apr-2026 13:32:25 DEBUG GEOparse - PLATFORM: GPL23159
DEBUG:GEOparse:PLATFORM: GPL23159
10-Apr-2026 13:32:29 WARNING GEOTypes - Detected duplicated columns in d GPL23159. Correcting.


10-Apr-2026 13:32:29 DEBUG GEOparse - SAMPLE: GSM6559856
DEBUG:GEOparse:SAMPLE: GSM6559856
10-Apr-2026 13:32:29 DEBUG GEOparse - SAMPLE: GSM6559857
DEBUG:GEOparse:SAMPLE: GSM6559857
10-Apr-2026 13:32:29 DEBUG GEOparse - SAMPLE: GSM6559858
DEBUG:GEO

Dataset: GSE212865
Title: Dynamics of gene expression profiling by microarrays and identification of high-risk patients for severe COVID-19 [Array]
Samples: 137
Expression Matrix Shape: (27189, 137)
✅ Saved: DEG.csv
